# AAAI-27 Research Paper: "Privacy-Preserving Federated Learning for Clinical Text Classification"
## Centralized HMLTC Training, Evaluation & Handoff Notebook (Single Source Deliverable)

**NLP Modality Subgroup:** Husain, Shreya, Arpit  
**Target Handoff:** Federated Learning Team (Harshit, Ratan)

---
### Overview
This notebook is the single self-contained deliverable for the **Hierarchical Multi-Label Clinical Text Classification (HMLTC)** modality. It contains the complete pipeline logic (data loading, preprocessing, ontology hierarchy, PubMedBERT model, loss functions, training loop, metrics evaluation, and inference API) without requiring external dependencies.

---


### ⚙️ Section 1 — Configuration Management


In [ ]:
import os
import sys
import json
import math
import random
import re
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional, Set, Union
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class PipelineConfig:
    """Centralized configuration management for HMLTC pipeline."""
    def __init__(
        self,
        data_dir: Union[str, Path] = "..",
        output_dir: Union[str, Path] = "training_outputs",
        model_name: str = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        batch_size: int = 8,
        max_seq_length: int = 512,
        learning_rate: float = 2e-5,
        weight_decay: float = 0.01,
        num_epochs: int = 1,
        loss_type: str = "combined",
        focal_gamma: float = 2.0,
        focal_alpha: float = 0.25,
        label_smoothing: float = 0.05,
        hierarchy_penalty_weight: float = 0.1,
        grad_accum_steps: int = 2,
        max_grad_norm: float = 1.0,
        random_seed: int = 42,
        train_ratio: float = 0.7,
        val_ratio: float = 0.15,
        test_ratio: float = 0.15,
        classification_threshold: float = 0.5,
        dropout_rate: float = 0.2,
        use_label_attention: bool = False,
        device: Optional[str] = None,
        pin_memory: bool = False,
    ):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_seq_length = max_seq_length
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.num_epochs = num_epochs
        self.loss_type = loss_type
        self.focal_gamma = focal_gamma
        self.focal_alpha = focal_alpha
        self.label_smoothing = label_smoothing
        self.hierarchy_penalty_weight = hierarchy_penalty_weight
        self.grad_accum_steps = grad_accum_steps
        self.max_grad_norm = max_grad_norm
        self.random_seed = random_seed
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.classification_threshold = classification_threshold
        self.dropout_rate = dropout_rate
        self.use_label_attention = use_label_attention
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.pin_memory = pin_memory

    def summary(self) -> str:
        return (
            f"PipelineConfig(\n"
            f"  model_name={self.model_name!r},\n"
            f"  loss_type={self.loss_type!r},\n"
            f"  batch_size={self.batch_size},\n"
            f"  learning_rate={self.learning_rate},\n"
            f"  num_epochs={self.num_epochs},\n"
            f"  device={self.device!r}\n"
            f")"
        )

# Set seed function
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(data: Any, filepath: Union[str, Path]) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_json(filepath: Union[str, Path]) -> Any:
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

# Instantiate Config
config = PipelineConfig()
set_seed(config.random_seed)
output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(config.summary())


### 📝 Section 2 — Clinical Text Preprocessing


In [ ]:
def clean_text(text: str) -> str:
    """Basic whitespace normalization."""
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_clinical_text(text: str) -> str:
    """Advanced clinical text cleaner (de-identification placeholders, normalization)."""
    if not text:
        return ""
    text = re.sub(r"\[\*\*.*?\*\*\]", " ", text)  # remove MIMIC anonymization tags
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("✓ Preprocessing functions defined.")


### 🔌 Section 3 — Data Ingestion Layer (JSONL Data Plugin)


In [ ]:
class BaseDataPlugin:
    """Abstract base class for data plugins."""
    def load_records(self) -> List[Dict[str, Any]]:
        raise NotImplementedError

class JSONLDataPlugin(BaseDataPlugin):
    """Data plugin for loading JSON Lines dataset files."""
    def __init__(self, filepath: Union[str, Path], text_key: str = "text", clean: bool = False):
        self.filepath = Path(filepath)
        self.text_key = text_key
        self.clean = clean

    def load_records(self) -> List[Dict[str, Any]]:
        records = []
        with open(self.filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                text = item.get(self.text_key, "")
                if self.clean:
                    text = clean_clinical_text(text)

                parent_labels = item.get("parent_labels", [])
                child_labels = item.get("child_labels", [])
                labels = item.get("labels", parent_labels + child_labels)

                doc_id = item.get("doc_id", item.get("id", ""))
                patient_id = item.get("patient_id", doc_id)
                admission_id = item.get("admission_id", doc_id)

                records.append({
                    "doc_id": doc_id,
                    "text": text,
                    "labels": labels,
                    "parent_labels": parent_labels,
                    "child_labels": child_labels,
                    "patient_id": patient_id,
                    "admission_id": admission_id,
                    "split": item.get("split", "train"),
                    "site_id": item.get("site_id", 0),
                    "icd_codes": item.get("icd_codes", []),
                })
        return records

# Find dataset file
dataset_path = Path("dataset_transformed.jsonl")
if not dataset_path.exists():
    dataset_path = Path("../dataset_transformed.jsonl")

plugin = JSONLDataPlugin(dataset_path, clean=True)
all_records = plugin.load_records()

train_records = [r for r in all_records if r.get("split") == "train"]
val_records = [r for r in all_records if r.get("split") == "val"]
test_records = [r for r in all_records if r.get("split") == "test"]

if not val_records or not test_records:
    random.seed(config.random_seed)
    shuffled = all_records.copy()
    random.shuffle(shuffled)
    n_total = len(shuffled)
    n_train = int(n_total * config.train_ratio)
    n_val = int(n_total * config.val_ratio)
    train_records = shuffled[:n_train]
    val_records = shuffled[n_train:n_train + n_val]
    test_records = shuffled[n_train + n_val:]

print(f"Total records ingested : {len(all_records)}")
print(f"  Train split         : {len(train_records)}")
print(f"  Validation split    : {len(val_records)}")
print(f"  Test split          : {len(test_records)}")


### 🔍 Section 4 — Dataset Verification & Integrity Checks


In [ ]:
doc_ids = [r["doc_id"] for r in all_records]
unique_doc_ids = set(doc_ids)
empty_docs = [r for r in all_records if not r["text"].strip()]
missing_labels = [r for r in all_records if not r["labels"]]

all_parent_labels = [p for r in all_records for p in r.get("parent_labels", [])]
all_child_labels = [c for r in all_records for c in r.get("child_labels", [])]
text_lengths = [len(r["text"].split()) for r in all_records]

stats = {
    "total_records": len(all_records),
    "train_size": len(train_records),
    "val_size": len(val_records),
    "test_size": len(test_records),
    "unique_doc_ids": len(unique_doc_ids),
    "duplicate_doc_ids": len(all_records) - len(unique_doc_ids),
    "empty_docs_count": len(empty_docs),
    "missing_labels_count": len(missing_labels),
    "unique_parent_labels": len(set(all_parent_labels)),
    "unique_child_labels": len(set(all_child_labels)),
    "avg_text_length_words": float(sum(text_lengths) / max(len(text_lengths), 1)),
    "max_text_length_words": max(text_lengths) if text_lengths else 0,
    "parent_label_counts": dict(Counter(all_parent_labels)),
    "child_label_counts": dict(Counter(all_child_labels)),
}

print("=" * 65)
print("  DATASET VERIFICATION REPORT")
print("=" * 65)
print(f"  Total Records      : {stats['total_records']}")
print(f"  Unique Doc IDs     : {stats['unique_doc_ids']}")
print(f"  Duplicate Doc IDs  : {stats['duplicate_doc_ids']}")
print(f"  Empty Documents    : {stats['empty_docs_count']}")
print(f"  Missing Labels     : {stats['missing_labels_count']}")
print(f"  Parent Labels      : {stats['unique_parent_labels']}")
print(f"  Child Labels       : {stats['unique_child_labels']}")
print(f"  Avg Doc Length     : {stats['avg_text_length_words']:.1f} words")
print(f"  Max Doc Length     : {stats['max_text_length_words']} words")
print("=" * 65)

assert stats["empty_docs_count"] == 0, "ERROR: Empty documents found!"
assert stats["missing_labels_count"] == 0, "ERROR: Documents with missing labels found!"
assert stats["duplicate_doc_ids"] == 0, f"ERROR: {stats['duplicate_doc_ids']} duplicate doc IDs found!"

save_json(stats, output_dir / "dataset_statistics.json")
print("✓ Dataset verification passed with 0 errors.")


### 🌲 Section 5 — Coding System Hierarchy Builder


In [ ]:
class CodingSystemHierarchy:
    """Represents parent-child relationships for hierarchical multi-label classification."""
    def __init__(self, parent_to_children: Dict[str, List[str]], coding_system: str = "custom"):
        self.parent_to_children = {p: list(c) for p, c in parent_to_children.items()}
        self.coding_system = coding_system
        self.child_to_parent = {}
        for p, children in self.parent_to_children.items():
            for c in children:
                self.child_to_parent[c] = p

        self.parents = sorted(list(self.parent_to_children.keys()))
        self.children = sorted(list(self.child_to_parent.keys()))
        self.all_labels = sorted(list(set(self.parents) | set(self.children)))

    @classmethod
    def from_mapping(cls, mapping: Dict[str, List[str]], coding_system: str = "custom") -> "CodingSystemHierarchy":
        return cls(mapping, coding_system=coding_system)

    @classmethod
    def from_json(cls, filepath: Union[str, Path]) -> "CodingSystemHierarchy":
        data = load_json(filepath)
        return cls(data.get("parent_to_children", {}), coding_system=data.get("coding_system", "custom"))

    def to_json(self, filepath: Union[str, Path]) -> None:
        save_json({
            "coding_system": self.coding_system,
            "parent_to_children": self.parent_to_children,
            "child_to_parent": self.child_to_parent,
            "num_parents": len(self.parents),
            "num_children": len(self.children),
            "all_labels": self.all_labels,
        }, filepath)

    @property
    def num_labels(self) -> int:
        return len(self.all_labels)

    @property
    def num_parents(self) -> int:
        return len(self.parents)

    @property
    def num_children(self) -> int:
        return len(self.children)

    def summary(self) -> str:
        return f"CodingSystemHierarchy(system='{self.coding_system}', parents={self.num_parents}, children={self.num_children}, total={self.num_labels})"

# 3-Tier Hierarchy Resolution Strategy
hierarchy_file = output_dir / "label_hierarchy.json"

if hasattr(plugin, "get_explicit_hierarchy") and plugin.get_explicit_hierarchy():
    print("✓ Strategy 1: Using explicit hierarchy from dataset plugin.")
    hierarchy = CodingSystemHierarchy.from_mapping(plugin.get_explicit_hierarchy(), coding_system="custom")

elif hierarchy_file.exists():
    print(f"✓ Strategy 2: Loading hierarchy from {hierarchy_file}")
    hierarchy = CodingSystemHierarchy.from_json(hierarchy_file)

else:
    print("✓ Strategy 3: Dynamic extraction from dataset co-occurrences...")
    child_to_parent_counts = {}
    for r in all_records:
        parents = r.get("parent_labels", [])
        children = r.get("child_labels", [])
        for c in children:
            if c not in child_to_parent_counts:
                child_to_parent_counts[c] = Counter()
            for p in parents:
                child_to_parent_counts[c][p] += 1

    p2c = {}
    for c, p_counter in child_to_parent_counts.items():
        if p_counter:
            best_parent = p_counter.most_common(1)[0][0]
            if best_parent not in p2c:
                p2c[best_parent] = []
            p2c[best_parent].append(c)

    hierarchy = CodingSystemHierarchy.from_mapping(p2c, coding_system="dataset_resolved")
    assert hierarchy.num_parents > 0 and hierarchy.num_children > 0, "Hierarchy validation error!"

hierarchy.to_json(hierarchy_file)
print(hierarchy.summary())


### 🏷️ Section 6 — Hierarchical Multi-Hot Label Encoder


In [ ]:
class HierarchicalLabelEncoder:
    """Encodes string label lists into multi-hot binary vectors with parent consistency."""
    def __init__(self, hierarchy: CodingSystemHierarchy, enforce_consistency: bool = True):
        self.hierarchy = hierarchy
        self.enforce_consistency = enforce_consistency
        self.label_names = list(hierarchy.all_labels)
        self.num_labels = len(self.label_names)
        self.label_to_idx = {name: idx for idx, name in enumerate(self.label_names)}
        self.idx_to_label = {idx: name for idx, name in enumerate(self.label_names)}

    def encode_single(self, labels: List[str]) -> np.ndarray:
        vec = np.zeros(self.num_labels, dtype=np.float32)
        target_labels = set(labels)
        if self.enforce_consistency:
            for l in list(target_labels):
                if l in self.hierarchy.child_to_parent:
                    parent = self.hierarchy.child_to_parent[l]
                    target_labels.add(parent)
        for l in target_labels:
            if l in self.label_to_idx:
                vec[self.label_to_idx[l]] = 1.0
        return vec

    def encode(self, labels_batch: List[List[str]]) -> np.ndarray:
        return np.vstack([self.encode_single(lbls) for lbls in labels_batch])

    def decode_single(self, vec: np.ndarray, threshold: float = 0.5) -> List[str]:
        return [self.idx_to_label[i] for i, val in enumerate(vec) if val >= threshold]

    def decode(self, matrix: np.ndarray, threshold: float = 0.5) -> List[List[str]]:
        return [self.decode_single(row, threshold) for row in matrix]

    def save(self, filepath: Union[str, Path]) -> None:
        save_json({
            "label_names": self.label_names,
            "label_to_idx": self.label_to_idx,
            "enforce_consistency": self.enforce_consistency,
        }, filepath)

label_encoder = HierarchicalLabelEncoder(hierarchy, enforce_consistency=True)
encoder_file = output_dir / "label_encoder.json"
label_encoder.save(encoder_file)
print(f"✓ Label Encoder initialized with {label_encoder.num_labels} total labels.")


### 📦 Section 7 — PyTorch Clinical Text Dataset & DataLoaders


In [ ]:
from transformers import AutoTokenizer

class ClinicalTextDataset(Dataset):
    """PyTorch Dataset wrapping clinical text and multi-hot labels."""
    def __init__(self, texts: List[str], labels_matrix: np.ndarray, tokenizer: Any, max_length: int = 512):
        self.texts = texts
        self.labels_matrix = labels_matrix
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = self.texts[idx]
        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels_matrix[idx], dtype=torch.float32),
        }
        if "token_type_ids" in encoded:
            item["token_type_ids"] = encoded["token_type_ids"].squeeze(0)
        return item

    @classmethod
    def from_records(cls, records: List[Dict[str, Any]], label_encoder: HierarchicalLabelEncoder, tokenizer: Any, max_length: int = 512) -> "ClinicalTextDataset":
        texts = [r["text"] for r in records]
        labels_batch = [r["labels"] for r in records]
        labels_matrix = label_encoder.encode(labels_batch)
        return cls(texts, labels_matrix, tokenizer, max_length=max_length)

# Build Tokenizer & Loaders
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
train_dataset = ClinicalTextDataset.from_records(train_records, label_encoder, tokenizer, max_length=config.max_seq_length)
val_dataset = ClinicalTextDataset.from_records(val_records, label_encoder, tokenizer, max_length=config.max_seq_length)
test_dataset = ClinicalTextDataset.from_records(test_records, label_encoder, tokenizer, max_length=config.max_seq_length)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=config.pin_memory)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, pin_memory=config.pin_memory)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, pin_memory=config.pin_memory)

print(f"✓ Train DataLoader : {len(train_loader)} batches ({len(train_dataset)} samples)")
print(f"✓ Val DataLoader   : {len(val_loader)} batches ({len(val_dataset)} samples)")
print(f"✓ Test DataLoader  : {len(test_loader)} batches ({len(test_dataset)} samples)")


### 🤖 Section 8 — PubMedBERT Classifier Architecture


In [ ]:
from transformers import AutoModel, AutoConfig

class ClinicalHMLTCModel(nn.Module):
    """BiomedBERT backbone + multi-label classification head."""
    def __init__(self, model_name: str, num_labels: int, dropout_rate: float = 0.2, use_label_attention: bool = False):
        super().__init__()
        self.model_name = model_name
        self.num_labels = num_labels
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, token_type_ids: Optional[torch.Tensor] = None, labels: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**kwargs)
        cls_output = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(cls_output)
        logits = self.classifier(pooled)
        return {"logits": logits, "probabilities": torch.sigmoid(logits)}

model = ClinicalHMLTCModel(config.model_name, label_encoder.num_labels, dropout_rate=config.dropout_rate).to(config.device)
print(f"✓ PubMedBERT model initialized on {config.device}")


### ⚖️ Section 9 — Focal & Hierarchical Consistency Loss Functions


In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss for multi-label class imbalance."""
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
            focal_weight = focal_weight * alpha_weight
        return (focal_weight * bce).mean()

class HierarchicalConsistencyLoss(nn.Module):
    """Penalizes violations where a child label is predicted positive but its parent is predicted negative."""
    def __init__(self, child_to_parent: Dict[str, str], label_to_idx: Dict[str, int]):
        super().__init__()
        pairs = []
        for c, p in child_to_parent.items():
            if c in label_to_idx and p in label_to_idx:
                pairs.append((label_to_idx[c], label_to_idx[p]))
        self.pairs = pairs

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        if not self.pairs:
            return torch.tensor(0.0, device=logits.device)
        probs = torch.sigmoid(logits)
        loss = 0.0
        for c_idx, p_idx in self.pairs:
            violation = F.relu(probs[:, c_idx] - probs[:, p_idx])
            loss += (violation ** 2).mean()
        return loss / len(self.pairs)

class CombinedHMLTCLoss(nn.Module):
    """Combines Focal Loss and Hierarchical Consistency Loss."""
    def __init__(self, focal_gamma: float = 2.0, focal_alpha: float = 0.25, hierarchy_weight: float = 0.1, child_to_parent: Optional[Dict[str, str]] = None, label_to_idx: Optional[Dict[str, int]] = None):
        super().__init__()
        self.focal = FocalLoss(gamma=focal_gamma, alpha=focal_alpha)
        self.hierarchy_weight = hierarchy_weight
        self.hierarchical = HierarchicalConsistencyLoss(child_to_parent or {}, label_to_idx or {}) if child_to_parent and label_to_idx else None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        f_loss = self.focal(logits, targets)
        if self.hierarchical is not None and self.hierarchy_weight > 0:
            h_loss = self.hierarchical(logits)
            return f_loss + self.hierarchy_weight * h_loss
        return f_loss

def build_loss(loss_type: str = "combined", **kwargs) -> nn.Module:
    if loss_type == "focal":
        return FocalLoss(gamma=kwargs.get("focal_gamma", 2.0), alpha=kwargs.get("focal_alpha", 0.25))
    elif loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    else:
        return CombinedHMLTCLoss(
            focal_gamma=kwargs.get("focal_gamma", 2.0),
            focal_alpha=kwargs.get("focal_alpha", 0.25),
            hierarchy_weight=kwargs.get("hierarchy_penalty_weight", 0.1),
            child_to_parent=kwargs.get("child_to_parent"),
            label_to_idx=kwargs.get("label_to_idx"),
        )

loss_fn = build_loss(
    config.loss_type,
    focal_gamma=config.focal_gamma,
    focal_alpha=config.focal_alpha,
    hierarchy_penalty_weight=config.hierarchy_penalty_weight,
    child_to_parent=hierarchy.child_to_parent,
    label_to_idx=label_encoder.label_to_idx,
)
print(f"✓ Loss constructed: {loss_fn}")


### 🚀 Section 10 — Training & Validation Engine


In [ ]:
def validate(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, device: str, threshold: float = 0.5) -> Tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            out = model(input_ids, attention_mask, token_type_ids=token_type_ids)
            loss = loss_fn(out["logits"], labels)
            total_loss += loss.item() * input_ids.size(0)

            probs = out["probabilities"].cpu().numpy()
            preds = (probs >= threshold).astype(np.float32)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    val_loss = total_loss / len(loader.dataset)
    return val_loss, np.vstack(all_preds), np.vstack(all_labels)

def fit(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module, device: str, num_epochs: int = 1, grad_accum_steps: int = 1, max_grad_norm: float = 1.0, threshold: float = 0.5) -> Dict[str, List[float]]:
    history = {"train_loss": [], "val_loss": []}
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            out = model(input_ids, attention_mask, token_type_ids=token_type_ids)
            loss = loss_fn(out["logits"], labels) / grad_accum_steps
            loss.backward()

            if step % grad_accum_steps == 0 or step == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item() * grad_accum_steps * input_ids.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss, _, _ = validate(model, val_loader, loss_fn, device, threshold)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch}/{num_epochs} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return history

from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

print(f"Starting training pass for {config.num_epochs} epoch(s)...")
history = fit(model, train_loader, val_loader, optimizer, loss_fn, config.device, num_epochs=config.num_epochs)


### 📊 Section 11 — Evaluation Metrics Suite


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def evaluate_multilabel(y_true: np.ndarray, y_pred: np.ndarray, label_names: Optional[List[str]] = None, model_name: str = "HMLTC", print_report: bool = True) -> Dict[str, float]:
    f1_micro = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    f1_macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_weighted = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    prec_micro = float(precision_score(y_true, y_pred, average="micro", zero_division=0))
    rec_micro = float(recall_score(y_true, y_pred, average="micro", zero_division=0))

    metrics = {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_micro": prec_micro,
        "recall_micro": rec_micro,
    }

    if print_report:
        print("=" * 65)
        print(f"  MULTILABEL EVALUATION REPORT ({model_name})")
        print("=" * 65)
        print(f"  F1 Micro       : {f1_micro:.4f}")
        print(f"  F1 Macro       : {f1_macro:.4f}")
        print(f"  F1 Weighted    : {f1_weighted:.4f}")
        print(f"  Precision Micro: {prec_micro:.4f}")
        print(f"  Recall Micro   : {rec_micro:.4f}")
        print("=" * 65)

    return metrics

def check_hierarchical_consistency(y_pred: np.ndarray, label_names: List[str], child_to_parent: Dict[str, str]) -> Dict[str, Any]:
    name_to_idx = {name: idx for idx, name in enumerate(label_names)}
    total_checks = 0
    violations = 0

    for sample in y_pred:
        for c, p in child_to_parent.items():
            if c in name_to_idx and p in name_to_idx:
                c_idx, p_idx = name_to_idx[c], name_to_idx[p]
                if sample[c_idx] == 1.0:
                    total_checks += 1
                    if sample[p_idx] != 1.0:
                        violations += 1

    rate = (1.0 - (violations / max(total_checks, 1))) if total_checks > 0 else 1.0
    return {"total_active_child_predictions": total_checks, "violations": violations, "consistency_rate": float(rate)}

val_loss, test_preds, test_labels = validate(model, test_loader, loss_fn, config.device, threshold=config.classification_threshold)
test_metrics = evaluate_multilabel(test_labels, test_preds, label_encoder.label_names, model_name="PubMedBERT-Centralized")
h_metrics = check_hierarchical_consistency(test_preds, label_encoder.label_names, hierarchy.child_to_parent)
test_metrics["hierarchy_consistency"] = h_metrics

print(f"✓ Test Hierarchy Consistency Rate: {h_metrics['consistency_rate']:.4%}")


### 💾 Section 12 — Deliverables & Checkpoint Export


In [ ]:
def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, metrics: Dict[str, Any], filepath: Union[str, Path]) -> None:
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
    }, filepath)

checkpoint_path = output_dir / "model_checkpoint.pt"
save_checkpoint(model, optimizer, config.num_epochs, test_metrics, checkpoint_path)

tokenizer_dir = output_dir / "tokenizer"
tokenizer.save_pretrained(tokenizer_dir)

save_json({
    "model_name": config.model_name,
    "max_seq_length": config.max_seq_length,
    "batch_size": config.batch_size,
    "learning_rate": config.learning_rate,
    "loss_type": config.loss_type,
    "classification_threshold": config.classification_threshold,
    "num_labels": label_encoder.num_labels,
}, output_dir / "pipeline_config.json")

save_json(test_metrics, output_dir / "metrics.json")
save_json({
    "model": config.model_name,
    "num_epochs": config.num_epochs,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "test_f1_micro": test_metrics.get("f1_micro", 0.0),
    "test_f1_macro": test_metrics.get("f1_macro", 0.0),
    "consistency_rate": h_metrics.get("consistency_rate", 0.0),
}, output_dir / "training_summary.json")

print("✓ Complete handoff bundle successfully saved to:", output_dir)


### 🔮 Section 13 — Inference API & Federated Handoff Wrapper


In [ ]:
class FederatedNLPClient:
    """Federated Learning client wrapper providing get/set weights and inference APIs."""
    def __init__(self, config: PipelineConfig, hierarchy: CodingSystemHierarchy, label_encoder: HierarchicalLabelEncoder):
        self.config = config
        self.hierarchy = hierarchy
        self.label_encoder = label_encoder
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        self.model = ClinicalHMLTCModel(config.model_name, label_encoder.num_labels, dropout_rate=config.dropout_rate).to(config.device)
        self.model.eval()

    def get_weights(self) -> Dict[str, torch.Tensor]:
        return {k: v.cpu().clone() for k, v in self.model.state_dict().items()}

    def set_weights(self, weights: Dict[str, torch.Tensor]) -> None:
        self.model.load_state_dict(weights)

    def predict(self, texts: List[str]) -> np.ndarray:
        self.model.eval()
        cleaned_texts = [clean_clinical_text(t) for t in texts]
        encoded = self.tokenizer(
            cleaned_texts,
            max_length=self.config.max_seq_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(self.config.device)
        attention_mask = encoded["attention_mask"].to(self.config.device)
        token_type_ids = encoded.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(self.config.device)

        with torch.no_grad():
            out = self.model(input_ids, attention_mask, token_type_ids=token_type_ids)
            return out["probabilities"].cpu().numpy()

    def predict_labels(self, texts: List[str], threshold: Optional[float] = None) -> List[List[str]]:
        t = threshold if threshold is not None else self.config.classification_threshold
        probs = self.predict(texts)
        return self.label_encoder.decode(probs, threshold=t)

client = FederatedNLPClient(config, hierarchy, label_encoder)
client.set_weights(model.state_dict())

sample_texts = [
    "Participant presented with mild cognitive impairment and underwent lumbar puncture for CSF biomarker testing.",
    "Data collection protocols and data dictionary standards for ADNI biofluid sample inventory.",
]

print("Inference Demo:")
for t in sample_texts:
    predicted_labels = client.predict_labels([t])[0]
    print(f"\nText snippet : '{t[:70]}...'")
    print(f"Predicted    : {predicted_labels}")


### 🏁 Section 14 — Final Deliverables Verification Checklist


In [ ]:
required_artifacts = [
    output_dir / "model_checkpoint.pt",
    output_dir / "tokenizer" / "tokenizer_config.json",
    output_dir / "pipeline_config.json",
    output_dir / "label_hierarchy.json",
    output_dir / "label_encoder.json",
    output_dir / "metrics.json",
    output_dir / "dataset_statistics.json",
    output_dir / "training_summary.json",
]

print("=" * 65)
print("  FINAL DELIVERABLES VERIFICATION CHECKLIST")
print("=" * 65)

all_passed = True
for artifact in required_artifacts:
    exists = artifact.exists()
    status = "✓ PASSED" if exists else "❌ FAILED"
    rel_path = artifact.relative_to(output_dir.parent) if artifact.is_relative_to(output_dir.parent) else artifact
    print(f"  [{status}] {rel_path}")
    if not exists:
        all_passed = False

print("=" * 65)
assert all_passed, "Deliverables Verification Failed! Missing required artifact files."

print("\n" + "★" * 65)
print("  ✓ NLP Pipeline Deliverables Verified & Ready for Federated Learning Team")
print("★" * 65)
